## using best practice
1. Preprocessing and datacleaning
2. Train Test Split
3. BOW,TFIDF, or Word2Vec
4. Train with ML Algos

In [2]:
!wget "https://raw.githubusercontent.com/krishnaik06/Complete-Data-Science-With-Machine-Learning-And-NLP-2024/main/26-CompleteNLP%20For%20Machine%20Learning/Practicals/Kindle%20Reviews/all_kindle_review.csv"

import pandas as pd

df = pd.read_csv("all_kindle_review.csv")
df.head()

--2026-04-26 11:33:10--  https://raw.githubusercontent.com/krishnaik06/Complete-Data-Science-With-Machine-Learning-And-NLP-2024/main/26-CompleteNLP%20For%20Machine%20Learning/Practicals/Kindle%20Reviews/all_kindle_review.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 8593622 (8.2M) [text/plain]
Saving to: ‘all_kindle_review.csv’

all_kindle_review.c 100%[===================>]   8.20M  --.-KB/s    in 0.09s   

2026-04-26 11:33:10 (92.7 MB/s) - ‘all_kindle_review.csv’ saved [8593622/8593622]



,Unnamed: 0.1,Unnamed: 0,asin,helpful,rating,reviewText,reviewTime,reviewerID,reviewerName,summary,unixReviewTime
0,0,11539,B0033UV8HI,"[8, 10]",3,"Jace Rankin may be short, but he's nothing to ...","09 2, 2010",A3HHXRELK8BHQG,Ridley,Entertaining But Average,1283385600
1,1,5957,B002HJV4DE,"[1, 1]",5,Great short read. I didn't want to put it dow...,"10 8, 2013",A2RGNZ0TRF578I,Holly Butler,Terrific menage scenes!,1381190400
2,2,9146,B002ZG96I4,"[0, 0]",3,I'll start by saying this is the first of four...,"04 11, 2014",A3S0H2HV6U1I7F,Merissa,Snapdragon Alley,1397174400
3,3,7038,B002QHWOEU,"[1, 3]",3,Aggie is Angela Lansbury who carries pocketboo...,"07 5, 2014",AC4OQW3GZ919J,Cleargrace,very light murder cozy,1404518400
4,4,1776,B001A06VJ8,"[0, 1]",4,I did not expect this type of book to be in li...,"12 31, 2012",A3C9V987IQHOQD,Rjostler,Book,1356912000


In [3]:
## extract only the columns that on which we have to perform the sentimental anaylsis
df=df[['reviewText','rating']]

In [4]:
df.head()

,reviewText,rating
0,"Jace Rankin may be short, but he's nothing to ...",3
1,Great short read. I didn't want to put it dow...,5
2,I'll start by saying this is the first of four...,3
3,Aggie is Angela Lansbury who carries pocketboo...,3
4,I did not expect this type of book to be in li...,4


In [5]:
df.shape

(12000, 2)

In [6]:
df.isnull().sum()

,0
reviewText,0
rating,0


In [7]:
df['rating'].unique()

array([3, 5, 4, 2, 1])

In [8]:
df['rating'].value_counts()

,count
rating,
5,3000
4,3000
3,2000
2,2000
1,2000


#### 1. Preprocessing and Cleaning

In [9]:
## positive --> 1
## negative --> 0
df['rating'] = df['rating'].apply(lambda x: 0 if x < 2.5 else 1)

In [10]:
df['rating'].value_counts()

,count
rating,
1,8000
0,4000


In [27]:
## 1. Lower all the cases
df['reviewText']=df['reviewText'].str.lower()

In [13]:
## 2. Removing special characters
import re
df['reviewText'] = df['reviewText'].apply(lambda x : re.sub('[^a-z AZ 0-9]+','',x))


In [22]:
## Removing all the stopwords
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))
df['reviewText'] = df['reviewText'].apply(lambda x : " ".join([y for y in x.split() if y not in stop_words]))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [23]:
## Remove URLs
df['reviewText'] = df['reviewText'].apply(lambda x: re.sub(r'(http|https|ftp|ssh)://\S+', '', str(x)))

In [24]:
## Remove html tags
from bs4 import BeautifulSoup
df['reviewText'] = df['reviewText'].apply(lambda x : BeautifulSoup(x,'lxml').get_text())

In [28]:
## Remove the additional spavces
df['reviewText'] = df['reviewText'].apply(lambda x: " ".join(x.split()))

In [29]:
df.head()

,reviewText,rating
0,ace ankin may short nothing mess man hauled sa...,1
1,reat short read want put read one sitting sex ...,1
2,start saying first four book expecting 34concl...,1
3,aggie angela ansbury carry pocketbook instead ...,1
4,expect type book library pleased find price right,1


In [30]:
## Applying wordnet lemmatizer
nltk.download('wordnet')
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [31]:
def lemmatize_words(text):
  return " ".join([lemmatizer.lemmatize(word) for word in text.split()])

In [32]:
df['reviewText'] = df['reviewText'].apply(lambda x: lemmatize_words(x))

In [33]:
df.head()

,reviewText,rating
0,ace ankin may short nothing mess man hauled sa...,1
1,reat short read want put read one sitting sex ...,1
2,start saying first four book expecting 34concl...,1
3,aggie angela ansbury carry pocketbook instead ...,1
4,expect type book library pleased find price right,1


In [34]:
X = df['reviewText']
y = df['rating']

#### 2. Train Test Split

In [41]:
## train test split
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(df['reviewText'],df['rating'],test_size=0.20)

In [42]:
X_train

,reviewText
7532,ery rare find finishing book nfortunately deci...
11482,really wanted like well written short storyhow...
4850,elle aylor spent last year working eighty hour...
9705,woman abusive background find love lasting rel...
4810,tarting thriller book slows go seems lose orig...
...,...
3780,ot sure polyamorus storyline enjoy world autho...
10217,review novel main theme presented detailed pro...
5036,ot hard figure mystery written wellenough stil...
431,a story orna ames went pleasantly surprised wa...


In [43]:
X_train.shape , y_train.shape , X_test.shape , y_test.shape

((9600,), (9600,), (2400,), (2400,))

#### 3. BOW,TFIDF, or Word2Vec

In [44]:
## Applying BOW
from sklearn.feature_extraction.text import CountVectorizer
bow = CountVectorizer()

In [45]:
X_train_bow = bow.fit_transform(X_train).toarray()
X_test_bow = bow.transform(X_test).toarray()

In [46]:
## applying TFIDF
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()

X_train_tfidf = tfidf.fit_transform(X_train).toarray()
X_test_tfidf = tfidf.transform(X_test).toarray()

#### we will compare both TFIDF and BOW

In [47]:
X_train_bow

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

#### 4. Train with ML Algos --> Gaussian Naive Bias

In [48]:
## apply ml algo
from sklearn.naive_bayes import GaussianNB
nb_model_bow = GaussianNB().fit(X_train_bow,y_train)
nb_mode_tfidf = GaussianNB().fit(X_train_tfidf,y_train)

In [49]:
from sklearn.metrics import confusion_matrix,accuracy_score,classification_report
y_pred_bow = nb_model_bow.predict(X_test_bow)

In [50]:
y_pred_tfidf = nb_mode_tfidf.predict(X_test_tfidf)

In [51]:
print("BOW accuracy: ",accuracy_score(y_test,y_pred_bow))

BOW accuracy:  0.595


In [52]:
confusion_matrix(y_test,y_pred_bow)

array([[493, 294],
       [678, 935]])

In [53]:
print("TFIDF accuracy: ",accuracy_score(y_test,y_pred_tfidf))

TFIDF accuracy:  0.5983333333333334


In [54]:
confusion_matrix(y_test,y_pred_tfidf)

array([[488, 299],
       [665, 948]])

## As the accuracy is not good so we will try it with Word2Vec
- Word2Vec is better of this type of huge dataset
- using the Gensim library

In [56]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 20.4 MB/s eta 0:00:00


In [57]:
from nltk import sent_tokenize
from gensim.utils import simple_preprocess

In [59]:
X_train

,reviewText
7532,ery rare find finishing book nfortunately deci...
11482,really wanted like well written short storyhow...
4850,elle aylor spent last year working eighty hour...
9705,woman abusive background find love lasting rel...
4810,tarting thriller book slows go seems lose orig...
...,...
3780,ot sure polyamorus storyline enjoy world autho...
10217,review novel main theme presented detailed pro...
5036,ot hard figure mystery written wellenough stil...
431,a story orna ames went pleasantly surprised wa...


In [64]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [65]:
words = []
for sent in X_train:
  sent_token = sent_tokenize(sent)
  for word in sent_token:
    words.append(simple_preprocess(word))

In [66]:
words

[['ery',
  'rare',
  'find',
  'finishing',
  'book',
  'nfortunately',
  'decided',
  'one',
  'hold',
  'attention',
  'however',
  'best',
  'read',
  'searching',
  'amazon',
  'found',
  'book',
  'rather',
  'read',
  'put',
  'one',
  'decided',
  'hurry',
  'pick',
  'back',
  'finish',
  'last',
  'anytime',
  'soonhere',
  'allot',
  'page',
  'filled',
  'justuseless',
  'story',
  'telling',
  'little',
  'story',
  'mesh',
  'together',
  'ike',
  'different',
  'idea',
  'used',
  'real',
  'thought',
  'go',
  'together',
  'ou',
  'try',
  'put',
  'chemistry',
  'romance',
  'book',
  'reader',
  'actually',
  'feel',
  'itwell',
  'ori',
  'missed'],
 ['really',
  'wanted',
  'like',
  'well',
  'written',
  'short',
  'storyhoweverit',
  'sweet',
  'unbelievable',
  'go',
  'fantasy'],
 ['elle',
  'aylor',
  'spent',
  'last',
  'year',
  'working',
  'eighty',
  'hour',
  'week',
  'without',
  'single',
  'day',
  'hristmas',
  'time',
  'od',
  'going',
  'visit',

In [68]:
import gensim

In [69]:
## training the word2vec model from scratch
model = gensim.models.Word2Vec(words)

In [70]:
## to get all the vocabulary
model.wv.index_to_key

['book',
 'story',
 'read',
 'character',
 'one',
 'like',
 'good',
 'would',
 'really',
 'author',
 'get',
 'time',
 'love',
 'series',
 'reading',
 'much',
 'first',
 'well',
 'know',
 'short',
 'way',
 'even',
 'make',
 'could',
 'sex',
 'little',
 'want',
 'thing',
 'great',
 'two',
 'think',
 'find',
 'plot',
 'also',
 'end',
 'romance',
 'life',
 'go',
 'enjoyed',
 'scene',
 'see',
 'never',
 'take',
 'say',
 'ut',
 'bit',
 'woman',
 'written',
 'work',
 'found',
 'lot',
 'many',
 'give',
 'going',
 'thought',
 'novel',
 'year',
 'writing',
 'better',
 'liked',
 'feel',
 'another',
 'interesting',
 'hat',
 'and',
 'come',
 'got',
 'back',
 'enough',
 'made',
 'people',
 'man',
 'friend',
 'reader',
 'something',
 'review',
 'page',
 'part',
 'though',
 'need',
 'bad',
 'still',
 'indle',
 'loved',
 'free',
 'cant',
 'relationship',
 'world',
 'together',
 'star',
 'hot',
 'felt',
 'next',
 'keep',
 'enjoy',
 'recommend',
 'start',
 'hen',
 'put',
 'best',
 'main',
 'new',
 'word'

In [71]:
model.corpus_count

9600

#### using AvgWord2Vec on top of it

In [72]:
import numpy as np

def avg_word2vec(doc):
    vectors = [model.wv[word] for word in doc if word in model.wv.index_to_key]

    if len(vectors) == 0:
        return np.zeros(model.vector_size)

    return np.mean(vectors, axis=0)

In [73]:
!pip install tqdm

In [74]:
from tqdm import tqdm

In [75]:
## applying avgword2vec to entire sentence
X = []
for i in tqdm(range(len(words))):
  X.append(avg_word2vec(words[i]))

100%|██████████| 9600/9600 [00:39<00:00, 245.62it/s]


In [76]:
len(X)

9600

In [77]:
## Independent features
X_new = np.array(X)

In [81]:
X_new.shape

(9600, 100)

In [82]:
## 100 dimension vector
X_new[0].shape

(100,)

In [88]:
y_new = y_train

#### 4. Train with ML Algos --> Using Gaussian Naive Bayes with Word2Vec

In [89]:
from sklearn.naive_bayes import GaussianNB
nb_model_word2vec = GaussianNB().fit(X_new, y_new)

In [90]:
words_test = []
for sent in X_test:
    sent_token = sent_tokenize(sent)
    for word in sent_token:
        words_test.append(simple_preprocess(word))


X_test_word2vec = []
for i in tqdm(range(len(words_test))):
    X_test_word2vec.append(avg_word2vec(words_test[i]))

X_test_word2vec = np.array(X_test_word2vec)

100%|██████████| 2400/2400 [00:04<00:00, 544.55it/s]


In [91]:
from sklearn.metrics import accuracy_score, confusion_matrix
y_pred_word2vec = nb_model_word2vec.predict(X_test_word2vec)

print("Word2Vec accuracy:", accuracy_score(y_test, y_pred_word2vec))
print("Confusion Matrix for Word2Vec:")
print(confusion_matrix(y_test, y_pred_word2vec))

Word2Vec accuracy: 0.6729166666666667
Confusion Matrix for Word2Vec:
[[ 559  228]
 [ 557 1056]]
